In [21]:
from transformers import AutoTokenizer, LlamaForCausalLM
import torch
from translate import Translator

In [ ]:
model = LlamaForCausalLM.from_pretrained('New Folder/sangrah_model_3.4')
tokenizer = AutoTokenizer.from_pretrained('New Folder/sangrah_tokenizers')

# Move model to the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# Initialize the Translator
translator = Translator(from_lang='ne', to_lang='en')

In [53]:
# Example prompt
# input_text = "डा विपिन अधिकारी,संविधानविद् यो समस्या सरकारमा थियो। सरकारबाट संसद्मा पुग्यो। संसद्बाट सर्वोच्चमा पुगेपछि सर्वोच्च एउटा निर्णयमा पुग्यो। हुन नहुने जुन"
inputs = [
    "शिक्षा र स्वास्थ्य क्षेत्रमा सुधारको आवश्यकता, विशेष गरी ग्रामीण र हाशिएका क्षेत्रमा, र यी क्षेत्रहरूले दीर्घकालीन राष्ट्रिय विकासमा कसरी योगदान पुर्याउन सक्छन्?"
]

a = ""
i = 1
for input_text in inputs :
    a += f"{i}) Prompt in nepali: " + input_text
	# Tokenize and move input to the correct device
    df = tokenizer(input_text, return_tensors='pt', padding=True, return_attention_mask = True)
    # input_ids = input_ids.to(device)
    input_ids = df["input_ids"]
    attention_mask = df["attention_mask"]

    # Set pad_token_id and create attention_mask
    model.config.pad_token_id = model.config.eos_token_id
    # attention_mask = (input_ids != tokenizer.pad_token_id).long()

    # Generate text
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=150,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            temperature=0.1
        )

    # Decode the output
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    a += "\n\nGenerated Nepali Text: " + generated_text

    # Translate Nepali to English
    translated_input = translator.translate(input_text)
    translated_output = translator.translate(generated_text)

    a += "\n\nPrompt in English: " + translated_input + "\n\nOutput in Engilsh: " + translated_output + "\n\n\n\n"

    i += 1
    
with open("output.txt", 'w', encoding='utf-8') as file :
    file.write(a[:-14])

/home/sumeet/.local/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:598: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
